# Kaggle submission

Two submissions so the bought-vs-built gap can be read on the real private leaderboard: **combined** (built + bought) and **internal only** (built). All feature blocks (base, external, and the derived cross / pca / kmeans / autoencoder features) now cover train and test, so the full selected set is used. Uses tuned params from `best_params.json` (notebook 21) when present, else defaults.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import json
import pandas as pd
import lightgbm as lgb
import pyarrow.parquet as pq
from pathlib import Path
from src.data import RAW, INTERIM, BLOCKS, DERIVED, load_selected

X, y = load_selected()
SRC = ["application"] + BLOCKS + DERIVED                  # all now cover train + test
avail = set()
for name in SRC:
    avail |= set(pq.ParquetFile(INTERIM / f"{name}.parquet").schema.names)
cols = [c for c in X.columns if c in avail]              # base + external (test-available)
ext = [c for c in cols if "ext_source" in c.lower() or c.lower().startswith("ext_calc_")]
internal = [c for c in cols if c not in ext]
print(f"combined {len(cols)} feats | internal {len(internal)} | dropped train-only derived {len(X.columns) - len(cols)}")

bp = json.loads((INTERIM / "best_params.json").read_text()) if (INTERIM / "best_params.json").exists() else {}
P = {"n_estimators": 800, "learning_rate": 0.03, "num_leaves": 31, "min_child_samples": 50,
     "subsample": 0.8, "colsample_bytree": 0.7, "reg_lambda": 2.0, **bp.get("lgb", {})}
def gbm():
    return lgb.LGBMClassifier(**P, subsample_freq=1, n_jobs=-1, verbose=-1)

combined 441 feats | internal 427 | dropped train-only derived 0


## Build the test matrix

In [2]:
test = pd.read_csv(RAW / "application_test.csv", usecols=["SK_ID_CURR"]).set_index("SK_ID_CURR")
for name in SRC:
    a = [c for c in pq.ParquetFile(INTERIM / f"{name}.parquet").schema.names if c in cols]
    if a:
        test = test.join(pd.read_parquet(INTERIM / f"{name}.parquet", columns=["SK_ID_CURR"] + a).set_index("SK_ID_CURR"), how="left")
Xtest = test[cols]
OUT = Path("../data/submissions"); OUT.mkdir(parents=True, exist_ok=True)
Xtest.shape

(48744, 441)

## Submission 1 — combined (built + bought)

In [3]:
pred = gbm().fit(X[cols], y).predict_proba(Xtest[cols])[:, 1]
pd.DataFrame({"SK_ID_CURR": Xtest.index, "TARGET": pred}).to_csv(OUT / "submission_combined.csv", index=False)
print("wrote submission_combined.csv | pred mean", round(float(pred.mean()), 4))

wrote submission_combined.csv | pred mean 0.0703


## Submission 2 — internal only (built)

In [4]:
pred_i = gbm().fit(X[internal], y).predict_proba(Xtest[internal])[:, 1]
pd.DataFrame({"SK_ID_CURR": Xtest.index, "TARGET": pred_i}).to_csv(OUT / "submission_internal.csv", index=False)
print("wrote submission_internal.csv | pred mean", round(float(pred_i.mean()), 4))

wrote submission_internal.csv | pred mean 0.0705
